# CELL 1 — Introduction

This paired, 10-question pilot compares official open-weight `Qwen/Qwen3-1.7B` in FP16 with `Qwen/Qwen3-8B` in bitsandbytes NF4 4-bit on the repository's frozen GSM8K validation collection plan. Target hardware is one Kaggle Tesla T4 (16 GB); two GPUs are not treated as pooled memory. No paid API or LiteLLM provider is used. The plan, dataset, prompt, model revisions and local execution provenance are checked and saved. Outputs include per-model records, paired CSV, metrics, figures, error analysis, environment and manifest.

This is a small descriptive pilot, not a statistically powered comparison. Qwen's chat framing differs from the project's LiteLLM message framing. The pilot uses greedy decoding as an explicit experiment choice, not Qwen's recommended sampling recipe. Model availability, download size, Kaggle GPU availability and NF4 memory behavior must be confirmed during the real Kaggle smoke run. Results are not commercial-provider benchmark results and are never marked Phase 3A `scientific_eligible`.


## CELL 2 — Environment setup
Check existing versions first; install only missing/incompatible packages without changing the Kaggle PyTorch/CUDA build.

In [6]:
import importlib
import subprocess
import sys
from importlib import metadata

import torch
from packaging.version import Version

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. In Kaggle, select Accelerator: GPU (T4) and restart the session."
    )
if Version(torch.__version__.split("+")[0]) < Version("2.4"):
    raise RuntimeError(
        f"bitsandbytes requires PyTorch >=2.4; current torch={torch.__version__}. "
        "Select a compatible Kaggle image. This notebook will not change PyTorch or CUDA."
    )

# Install only missing/incompatible packages. --no-deps prevents pip from replacing
# Kaggle's PyTorch/CUDA; the relevant Transformers/HF transitive dependencies are
# checked explicitly below and installed individually only when absent/too old.
checks = {
    "transformers": ("4.51.0", "transformers>=4.51.0"),
    "accelerate": ("0", "accelerate"),
    "bitsandbytes": ("0.43.0", "bitsandbytes>=0.43.0"),
    "numpy": ("0", "numpy"),
    "matplotlib": ("0", "matplotlib"),
    "huggingface-hub": ("0.30.0", "huggingface-hub>=0.30.0"),
    "tokenizers": ("0.21.0", "tokenizers>=0.21.0"),
    "safetensors": ("0.4.3", "safetensors>=0.4.3"),
    "filelock": ("0", "filelock"),
    "packaging": ("0", "packaging"),
    "pyyaml": ("0", "pyyaml"),
    "regex": ("0", "regex"),
    "requests": ("0", "requests"),
    "tqdm": ("0", "tqdm"),
}
missing = []
for package, (minimum, requirement) in checks.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or (minimum != "0" and Version(installed) < Version(minimum)):
        missing.append(requirement)
    print(f"{package}: {installed or 'missing'} (minimum {minimum})")
if missing:
    print("Installing only listed missing/incompatible packages with --no-deps:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", *missing])

try:
    import accelerate
    import bitsandbytes
    import huggingface_hub
    import matplotlib
    import numpy as np
    import safetensors
    import tokenizers
    import transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    for dependency_module in ("filelock", "regex", "requests", "tqdm", "yaml"):
        importlib.import_module(dependency_module)
    if not callable(getattr(huggingface_hub, "hf_hub_download", None)):
        raise TypeError("huggingface_hub.hf_hub_download is unavailable")

    # Exercise the precise config API without loading weights or allocating a model.
    bnb_config_check = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
except Exception as exc:
    raise RuntimeError(
        "Kaggle inference dependency import/API check failed after targeted installs. "
        "The notebook did not alter PyTorch/CUDA. Inspect the exact import error and "
        "select a compatible Kaggle image before proceeding."
    ) from exc

print(
    {
        "python": sys.version,
        "torch": torch.__version__,
        "torch_cuda": torch.version.cuda,
        "transformers": transformers.__version__,
        "accelerate": accelerate.__version__,
        "bitsandbytes": bitsandbytes.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
        "tokenizers": tokenizers.__version__,
        "safetensors": safetensors.__version__,
        "huggingface_hub": huggingface_hub.__version__,
        "pytorch_cuda_modified": False,
        "model_load_test": "NOT_RUN until explicit smoke-test enablement",
    }
)

transformers: 5.20.2 (minimum 4.51.0)
accelerate: 1.15.0 (minimum 0)
bitsandbytes: 0.50.2 (minimum 0.43.0)
numpy: 2.1.3 (minimum 0)
matplotlib: 3.10.0 (minimum 0)
huggingface-hub: 1.33.0 (minimum 0.30.0)
tokenizers: 0.23.3 (minimum 0.21.0)
safetensors: 0.8.0 (minimum 0.4.3)
filelock: 4.2.4 (minimum 0)
packaging: 26.3 (minimum 0)
pyyaml: 6.0.3 (minimum 0)
regex: 2025.11.3 (minimum 0)
requests: 2.34.2 (minimum 0)
tqdm: 4.67.3 (minimum 0)
{'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]', 'torch': '2.11.0+cu130', 'torch_cuda': '13.0', 'transformers': '5.20.2', 'accelerate': '1.15.0', 'bitsandbytes': '0.50.2', 'numpy': '2.1.3', 'matplotlib': '3.10.0', 'tokenizers': '0.23.3', 'safetensors': '0.8.0', 'huggingface_hub': '1.33.0', 'pytorch_cuda_modified': False, 'model_load_test': 'NOT_RUN until explicit smoke-test enablement'}


## CELL 3 — GPU and VRAM inspection
Capture GPU 0 memory before model load. Select one GPU only; stop if CUDA is unavailable.

In [7]:
import json
import shutil
from pathlib import Path

GPU_INDEX = 0
if GPU_INDEX >= torch.cuda.device_count():
    raise RuntimeError(
        f"Requested GPU {GPU_INDEX}, but only {torch.cuda.device_count()} CUDA device(s) are visible."
    )
torch.cuda.set_device(GPU_INDEX)
free_bytes, total_bytes = torch.cuda.mem_get_info(GPU_INDEX)
try:
    import psutil

    cpu_ram = {
        "total_bytes": psutil.virtual_memory().total,
        "available_bytes": psutil.virtual_memory().available,
    }
except ImportError:
    cpu_ram = {"available": "psutil unavailable"}
disk = shutil.disk_usage("/kaggle/working")
GPU_BASELINE = {
    "name": torch.cuda.get_device_name(GPU_INDEX),
    "cuda": torch.version.cuda,
    "torch": torch.__version__,
    "gpu_count": torch.cuda.device_count(),
    "gpu_index": GPU_INDEX,
    "free_bytes": free_bytes,
    "total_bytes": total_bytes,
    "cpu_ram": cpu_ram,
    "working_disk_free_bytes": disk.free,
}
print(json.dumps(GPU_BASELINE, indent=2))

{
  "name": "Tesla T4",
  "cuda": "13.0",
  "torch": "2.11.0+cu130",
  "gpu_count": 2,
  "gpu_index": 0,
  "free_bytes": 15526068224,
  "total_bytes": 15636037632,
  "cpu_ram": {
    "total_bytes": 33658146816,
    "available_bytes": 31383789568
  },
  "working_disk_free_bytes": 20940562432
}


## CELL 4 — Reproducibility
Set all available RNG seeds and deterministic options. CUDA kernels, library implementations, and hardware may still introduce nondeterminism.

In [8]:
import random

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except (AttributeError, TypeError):
    pass
print(
    {
        "seed": SEED,
        "deterministic_algorithms_warn_only": True,
        "remaining_nondeterminism": "GPU kernels, quantized kernels, hardware and runtime versions may vary.",
    }
)

{'seed': 42, 'deterministic_algorithms_warn_only': True, 'remaining_nondeterminism': 'GPU kernels, quantized kernels, hardware and runtime versions may vary.'}


## CELL 5 — Dataset loading
Attach a Kaggle Dataset containing the repository's exact processed `train.jsonl`, `validation.jsonl`, `test.jsonl`, and `gsm8k.json` manifest. This notebook fails closed if files or hashes differ; official test records are validated but never selected.

In [9]:
from pathlib import Path

root = Path("/kaggle/input")

print("Các dataset đang được gắn:")
for path in root.iterdir():
    print(path)

print("\nCác file GSM8K tìm được:")
for path in root.rglob("*.jsonl"):
    print(path)

for path in root.rglob("gsm8k.json"):
    print(path)

Các dataset đang được gắn:

Các file GSM8K tìm được:


In [16]:
import os
import subprocess

repo = "/kaggle/working/ai-tutor-router"

subprocess.run(
    ["python", "-m", "experiments.dataset", "prepare",
     "--dataset", "gsm8k", "--seed", "42"],
    cwd=repo,
    check=True,
)

{
  "cross_split_overlap_count": 0,
  "dataset": "gsm8k",
  "dataset_content_sha256": "aa75c686332c09e12ce6412571b32b7efae779dfc8696d3448b29a45e5542e4b",
  "duplicate_pair_count": 1,
  "manifest": "data/manifests/gsm8k.json",
  "revision": "3101c7d5072418e28b9008a6636bde82a006892c",
  "sample_count": 8792,
  "split_counts": {
    "test": 1319,
    "train": 6726,
    "validation": 747
  }
}


CompletedProcess(args=['python', '-m', 'experiments.dataset', 'prepare', '--dataset', 'gsm8k', '--seed', '42'], returncode=0)

In [17]:
from pathlib import Path

repo = Path("/kaggle/working/ai-tutor-router")
for path in sorted((repo / "data").rglob("*")):
    if path.is_file():
        print(path.relative_to(repo))

data/manifests/gsm8k.json
data/processed/gsm8k/duplicates.json
data/processed/gsm8k/test.jsonl
data/processed/gsm8k/train.jsonl
data/processed/gsm8k/validation.jsonl
data/raw/gsm8k/source_manifest.json
data/raw/gsm8k/test.jsonl
data/raw/gsm8k/train.jsonl


In [18]:
import hashlib
import json
import subprocess
import sys

REPOSITORY = "https://github.com/NguyenDucThang-tb/ai-tutor-router.git"
REPOSITORY_BRANCH = "experiments/kaggle-t4-qwen3-pilot"
REPO_ROOT = Path("/kaggle/working/ai-tutor-router")
if not (REPO_ROOT / "experiments/kaggle_local.py").exists():
    if REPO_ROOT.exists():
        raise RuntimeError(
            f"{REPO_ROOT} exists but is not the expected repository. Remove/rename it and rerun this cell."
        )
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", REPOSITORY_BRANCH, REPOSITORY, str(REPO_ROOT)]
    )
sys.path.insert(0, str(REPO_ROOT))
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()

from experiments.kaggle_local import load_frozen_pilot, sample_ids_sha256

DATA_ROOT = Path("/kaggle/working/ai-tutor-router")
DATASET_DIR = DATA_ROOT / "data/processed/gsm8k"
MANIFEST_PATH = DATA_ROOT / "data/manifests/gsm8k.json"
PLAN_PATH = REPO_ROOT / "configs/phase3a_pilot.collection-plan.json"
required = [DATASET_DIR / f"{split}.jsonl" for split in ("train", "validation", "test")] + [
    MANIFEST_PATH,
    PLAN_PATH,
]
missing_paths = [str(path) for path in required if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Frozen GSM8K files are missing: "
        + ", ".join(missing_paths)
        + ". Attach the approved frozen dataset as Kaggle input named gsm8k-frozen-pilot; do not download or substitute another GSM8K snapshot."
    )
SAMPLES, DATASET_MANIFEST, COLLECTION_PLAN = load_frozen_pilot(
    DATASET_DIR, MANIFEST_PATH, PLAN_PATH
)
SAMPLE_IDS = [row.sample_id for row in SAMPLES]
assert len(SAMPLES) == 10 and len(set(SAMPLE_IDS)) == 10
print(
    json.dumps(
        {
            "repository_commit": REPO_COMMIT,
            "dataset_revision": DATASET_MANIFEST.revision,
            "dataset_sha256": DATASET_MANIFEST.dataset_content_sha256,
            "split": COLLECTION_PLAN["split"],
            "collection_plan_sha256": COLLECTION_PLAN["plan_sha256"],
            "sample_ids": SAMPLE_IDS,
            "sample_ids_sha256": sample_ids_sha256(SAMPLE_IDS),
            "count": len(SAMPLES),
        },
        indent=2,
    )
)

{
  "repository_commit": "25b56b88ff43728f94e66015ad03374cedf8a973",
  "dataset_revision": "3101c7d5072418e28b9008a6636bde82a006892c",
  "dataset_sha256": "aa75c686332c09e12ce6412571b32b7efae779dfc8696d3448b29a45e5542e4b",
  "split": "validation",
  "collection_plan_sha256": "1d5065acc16b50c1521422ec2a912ed28a10bd483d905054fbdd24aefd642493",
  "sample_ids": [
    "gsm8k-train-000279",
    "gsm8k-train-000288",
    "gsm8k-train-001457",
    "gsm8k-train-002506",
    "gsm8k-train-002716",
    "gsm8k-train-003336",
    "gsm8k-train-003798",
    "gsm8k-train-004196",
    "gsm8k-train-004440",
    "gsm8k-train-006450"
  ],
  "sample_ids_sha256": "077275a61e532506eab2d10c09d36960976253969380ca256af7b95cbcced5ed",
  "count": 10
}


## CELL 6 — Frozen prompting protocol
Reuse the Phase 3A system instruction verbatim. The GSM8K reference answer is not included in either model's messages. Qwen's official chat template adds model-specific framing; save its hash and rendered prompt hash per sample.

In [19]:
from experiments.prompts import PROMPT_SHA256, PROMPT_TEMPLATE, PROMPT_VERSION

CHAT_TEMPLATE_PROTOCOL = (
    "tokenizer.apply_chat_template(enable_thinking=False, add_generation_prompt=True)"
)
print(
    {
        "prompt_version": PROMPT_VERSION,
        "original_prompt_sha256": PROMPT_SHA256,
        "chat_template_protocol": CHAT_TEMPLATE_PROTOCOL,
        "framing_note": "Qwen tokenizer framing differs from the earlier LiteLLM provider path.",
    }
)

{'prompt_version': 'math-direct-v1', 'original_prompt_sha256': 'cbf5f86edcb3a0124cd39ecbf5350c9ba211e1689cf3109e250197a4a35ba560', 'chat_template_protocol': 'tokenizer.apply_chat_template(enable_thinking=False, add_generation_prompt=True)', 'framing_note': 'Qwen tokenizer framing differs from the earlier LiteLLM provider path.'}


## CELL 7 — Generation configuration
Batch size 1, max 1,024 input tokens, max 512 new tokens, seed 42, no retries/fallbacks, one greedy generation per model/question. Inputs over the limit are recorded as not executed with no truncation. Greedy decoding is a controlled pilot choice and differs from Qwen's recommended sampling settings.

In [20]:
import json
from pathlib import Path

MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 512
BATCH_SIZE = 1
SMALL_MODEL_ID = "Qwen/Qwen3-1.7B"
STRONG_MODEL_ID = "Qwen/Qwen3-8B"
SMALL_MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
STRONG_MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"
RUN_SMALL_SMOKE = True
RUN_STRONG_SMOKE = False
RUN_FULL_PILOT = False
SMOKE_TESTS_CONFIRMED = (
    False  # Set True only after both one-sample smoke runs succeed and are inspected.
)
OUTPUT_DIR = Path("/kaggle/working/outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
if RUN_FULL_PILOT:
    from experiments.kaggle_local import read_records

    if not SMOKE_TESTS_CONFIRMED:
        raise RuntimeError(
            "Run and inspect both smoke tests first, then explicitly set SMOKE_TESTS_CONFIRMED=True."
        )
    expected_smoke_id = SAMPLE_IDS[0] if "SAMPLE_IDS" in globals() else None
    for model_name in ("small", "strong"):
        smoke_records = read_records(CHECKPOINT_DIR / f"{model_name}_smoke.jsonl")
        if len(smoke_records) != 1 or smoke_records[0].get("status") != "completed":
            raise RuntimeError(
                f"A successful one-sample {model_name} smoke checkpoint is required before full execution."
            )
        if expected_smoke_id and smoke_records[0].get("sample_id") != expected_smoke_id:
            raise RuntimeError(
                f"The {model_name} smoke checkpoint does not match the frozen pilot's first sample."
            )
        expected_model = SMALL_MODEL_ID if model_name == "small" else STRONG_MODEL_ID
        expected_revision = SMALL_MODEL_REVISION if model_name == "small" else STRONG_MODEL_REVISION
        if (
            smoke_records[0].get("model_id") != expected_model
            or smoke_records[0].get("model_revision") != expected_revision
        ):
            raise RuntimeError(
                f"The {model_name} smoke checkpoint uses a different model or revision."
            )
        required_smoke_provenance = {
            "provider": "local_transformers",
            "scientific_eligible": False,
            "run_mode": "smoke",
            "dataset_revision": DATASET_MANIFEST.revision,
            "dataset_content_sha256": DATASET_MANIFEST.dataset_content_sha256,
            "collection_plan_sha256": COLLECTION_PLAN["plan_sha256"],
            "prompt_version": PROMPT_VERSION,
            "original_prompt_sha256": PROMPT_SHA256,
        }
        if any(
            smoke_records[0].get(key) != value for key, value in required_smoke_provenance.items()
        ):
            raise RuntimeError(
                f"The {model_name} smoke checkpoint provenance does not match the frozen pilot."
            )
print(
    json.dumps(
        {
            "small_model": SMALL_MODEL_ID,
            "small_revision": SMALL_MODEL_REVISION,
            "strong_model": STRONG_MODEL_ID,
            "strong_revision": STRONG_MODEL_REVISION,
            "max_input_tokens": MAX_INPUT_TOKENS,
            "max_new_tokens": MAX_NEW_TOKENS,
            "batch_size": BATCH_SIZE,
            "do_sample": False,
            "full_pilot_enabled": RUN_FULL_PILOT,
            "no_retry": True,
            "no_fallback": True,
        },
        indent=2,
    )
)

{
  "small_model": "Qwen/Qwen3-1.7B",
  "small_revision": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
  "strong_model": "Qwen/Qwen3-8B",
  "strong_revision": "b968826d9c46dd6066d109eabc6255188de91218",
  "max_input_tokens": 1024,
  "max_new_tokens": 512,
  "batch_size": 1,
  "do_sample": false,
  "full_pilot_enabled": false,
  "no_retry": true,
  "no_fallback": true
}


## CELL 8 — Small-model inference
FP16 single-GPU load. Smoke and full-run checkpoints are separate. A checkpointed success or failure is not automatically rerun after restart. Set the smoke flag first; full remains disabled until both smoke outputs have been reviewed.

In [21]:
import gc
import time

from transformers import AutoModelForCausalLM, AutoTokenizer

from experiments.kaggle_local import execute_samples


def build_inference(model_id, revision, quantized=False):
    cache = {}

    def infer(sample):
        if "load_error" in cache:
            error = RuntimeError(f"Model load failed: {cache['load_error']}")
            error.fatal_run_error = True
            raise error from cache["load_error"]
        if "model" not in cache:
            try:
                if quantized:
                    from transformers import BitsAndBytesConfig

                    quant_config = BitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.float16,
                        bnb_4bit_use_double_quant=False,
                    )
                    model = AutoModelForCausalLM.from_pretrained(
                        model_id,
                        revision=revision,
                        quantization_config=quant_config,
                        torch_dtype=torch.float16,
                        device_map={"": GPU_INDEX},
                        trust_remote_code=False,
                    )
                else:
                    model = AutoModelForCausalLM.from_pretrained(
                        model_id,
                        revision=revision,
                        torch_dtype=torch.float16,
                        device_map={"": GPU_INDEX},
                        trust_remote_code=False,
                    )
                tokenizer = AutoTokenizer.from_pretrained(
                    model_id, revision=revision, trust_remote_code=False
                )
                resolved = getattr(model.config, "_commit_hash", None)
                if resolved and resolved != revision:
                    raise RuntimeError(
                        f"Loaded model revision {resolved} does not match pinned {revision}"
                    )
                cache.update(
                    model=model, tokenizer=tokenizer, resolved_revision=resolved or revision
                )
                torch.cuda.reset_peak_memory_stats(GPU_INDEX)
            except Exception as exc:
                cache["load_error"] = exc
                error = RuntimeError(
                    f"Model load failed for {model_id}@{revision}: {type(exc).__name__}: {exc}"
                )
                error.fatal_run_error = True
                raise error from exc
        model, tokenizer = cache["model"], cache["tokenizer"]
        messages = [
            {"role": "system", "content": PROMPT_TEMPLATE},
            {"role": "user", "content": sample.question},
        ]
        rendered = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        encoded = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
        input_tokens = int(encoded["input_ids"].shape[-1])
        template_hash = hashlib.sha256((tokenizer.chat_template or "").encode()).hexdigest()
        prompt_hash = hashlib.sha256(rendered.encode()).hexdigest()
        base = {
            "input_tokens": input_tokens,
            "chat_template_sha256": template_hash,
            "effective_rendered_prompt_sha256": prompt_hash,
            "generation_config": {
                "do_sample": False,
                "max_new_tokens": MAX_NEW_TOKENS,
                "batch_size": BATCH_SIZE,
                "seed": SEED,
                "enable_thinking": False,
            },
        }
        if input_tokens > MAX_INPUT_TOKENS:
            return {
                **base,
                "status": "not_run_input_too_long",
                "model_output": None,
                "not_run_reason": "input exceeds 1024 tokens; prompt was not truncated",
            }
        device = next(model.parameters()).device
        encoded = {key: value.to(device) for key, value in encoded.items()}
        torch.cuda.synchronize(GPU_INDEX)
        torch.cuda.reset_peak_memory_stats(GPU_INDEX)
        start = time.perf_counter()
        try:
            generated = model.generate(**encoded, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)
            torch.cuda.synchronize(GPU_INDEX)
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                del encoded
                gc.collect()
                torch.cuda.empty_cache()
            raise
        elapsed = time.perf_counter() - start
        generated_ids = generated[0, input_tokens:]
        output = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return {
            **base,
            "status": "completed",
            "model_output": output,
            "output_tokens": int(generated_ids.numel()),
            "inference_latency_s": elapsed,
            "peak_gpu_allocated_bytes": torch.cuda.max_memory_allocated(GPU_INDEX),
            "peak_gpu_reserved_bytes": torch.cuda.max_memory_reserved(GPU_INDEX),
            "resolved_model_revision": cache["resolved_revision"],
            "quantization": (
                {"load_in_4bit": True, "type": "nf4", "compute_dtype": "float16"}
                if quantized
                else {"dtype": "float16"}
            ),
        }

    infer.cache = cache
    return infer


small_infer = build_inference(SMALL_MODEL_ID, SMALL_MODEL_REVISION)
small_enabled = RUN_SMALL_SMOKE or RUN_FULL_PILOT
small_samples = SAMPLES[:1] if RUN_SMALL_SMOKE and not RUN_FULL_PILOT else SAMPLES
small_path = CHECKPOINT_DIR / (
    "small_smoke.jsonl" if RUN_SMALL_SMOKE and not RUN_FULL_PILOT else "small_full.jsonl"
)
small_provenance = {
    "dataset_revision": DATASET_MANIFEST.revision,
    "dataset_content_sha256": DATASET_MANIFEST.dataset_content_sha256,
    "collection_plan_sha256": COLLECTION_PLAN["plan_sha256"],
    "prompt_version": PROMPT_VERSION,
    "original_prompt_sha256": PROMPT_SHA256,
    "run_mode": "smoke" if len(small_samples) == 1 else "full",
}
if RUN_FULL_PILOT:
    from datetime import UTC, datetime

    full_run_marker = {
        "schema_version": "kaggle-full-run-start-v1",
        "status": "started",
        "started_at_utc": datetime.now(UTC).isoformat(),
        "sample_ids": SAMPLE_IDS,
        "dataset_revision": DATASET_MANIFEST.revision,
        "dataset_content_sha256": DATASET_MANIFEST.dataset_content_sha256,
        "collection_plan_sha256": COLLECTION_PLAN["plan_sha256"],
        "prompt_version": PROMPT_VERSION,
        "original_prompt_sha256": PROMPT_SHA256,
        "small_model_id": SMALL_MODEL_ID,
        "small_model_revision": SMALL_MODEL_REVISION,
        "strong_model_id": STRONG_MODEL_ID,
        "strong_model_revision": STRONG_MODEL_REVISION,
    }
    marker_temp = CHECKPOINT_DIR / ".full_run_started.json.tmp"
    marker_temp.write_text(json.dumps(full_run_marker, indent=2, sort_keys=True) + "\n")
    marker_temp.replace(CHECKPOINT_DIR / "full_run_started.json")

small_checkpoint = execute_samples(
    small_samples,
    small_path,
    model_id=SMALL_MODEL_ID,
    model_revision=SMALL_MODEL_REVISION,
    run_enabled=small_enabled,
    infer_one=small_infer,
    provenance=small_provenance,
)
print(
    {
        "enabled": small_enabled,
        "checkpoint": str(small_path),
        "record_count": len(small_checkpoint),
        "status": [row["status"] for row in small_checkpoint],
    }
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

... (3 outputs left out)

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

{'enabled': True, 'checkpoint': '/kaggle/working/outputs/checkpoints/small_smoke.jsonl', 'record_count': 1, 'status': ['completed']}


## CELL 9 — GPU cleanup
Release the 1.7B model and CUDA tensors before loading Qwen3-8B. Confirm newly free memory; no two-GPU memory pooling or CPU fallback is used.

In [22]:
if small_enabled:
    small_infer.cache.clear()
    del small_infer
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize(GPU_INDEX)
free_after_small, total_after_small = torch.cuda.mem_get_info(GPU_INDEX)
print({"free_bytes_after_small_cleanup": free_after_small, "total_bytes": total_after_small})

{'free_bytes_after_small_cleanup': 15475736576, 'total_bytes': 15636037632}


## CELL 10 — Strong-model inference
Load Qwen3-8B using NF4 4-bit weights and FP16 compute on the selected GPU. An unsupported quantization configuration, OOM, download or generation failure is retained as a failure; no fallback to CPU/offload and no silent change to protocol.

In [23]:
strong_infer = build_inference(STRONG_MODEL_ID, STRONG_MODEL_REVISION, quantized=True)
strong_enabled = RUN_STRONG_SMOKE or RUN_FULL_PILOT
strong_samples = SAMPLES[:1] if RUN_STRONG_SMOKE and not RUN_FULL_PILOT else SAMPLES
strong_path = CHECKPOINT_DIR / (
    "strong_smoke.jsonl" if RUN_STRONG_SMOKE and not RUN_FULL_PILOT else "strong_full.jsonl"
)
strong_provenance = {
    "dataset_revision": DATASET_MANIFEST.revision,
    "dataset_content_sha256": DATASET_MANIFEST.dataset_content_sha256,
    "collection_plan_sha256": COLLECTION_PLAN["plan_sha256"],
    "prompt_version": PROMPT_VERSION,
    "original_prompt_sha256": PROMPT_SHA256,
    "run_mode": "smoke" if len(strong_samples) == 1 else "full",
    "quantization": {"load_in_4bit": True, "type": "nf4", "compute_dtype": "float16"},
}
strong_checkpoint = execute_samples(
    strong_samples,
    strong_path,
    model_id=STRONG_MODEL_ID,
    model_revision=STRONG_MODEL_REVISION,
    run_enabled=strong_enabled,
    infer_one=strong_infer,
    provenance=strong_provenance,
)
print(
    {
        "enabled": strong_enabled,
        "checkpoint": str(strong_path),
        "record_count": len(strong_checkpoint),
        "status": [row["status"] for row in strong_checkpoint],
    }
)

{'enabled': False, 'checkpoint': '/kaggle/working/outputs/checkpoints/strong_full.jsonl', 'record_count': 0, 'status': []}


## CELL 11 — Evaluate GSM8K correctness
Reuse the repository evaluator unchanged. Only completed model outputs are evaluated. Keep evaluator statuses and failures explicit; no output rewriting or local output promotion to provider data.

In [24]:
from evaluation.evaluator import EVALUATOR_VERSION, evaluate_answer
from experiments.kaggle_local import evaluate_records, read_records, write_records

small_all = read_records(CHECKPOINT_DIR / "small_full.jsonl")
strong_all = read_records(CHECKPOINT_DIR / "strong_full.jsonl")
small_evaluated = evaluate_records(small_all, evaluate_answer)
strong_evaluated = evaluate_records(strong_all, evaluate_answer)
write_records(OUTPUT_DIR / "small_results.jsonl", small_evaluated)
write_records(OUTPUT_DIR / "strong_results.jsonl", strong_evaluated)
print(
    {
        "evaluator_version": EVALUATOR_VERSION,
        "small_evaluated": sum(row["evaluation"] is not None for row in small_evaluated),
        "strong_evaluated": sum(row["evaluation"] is not None for row in strong_evaluated),
    }
)

{'evaluator_version': 'numeric-symbolic-v1', 'small_evaluated': 0, 'strong_evaluated': 0}


## CELL 12 — Metrics

Pair counts distinguish planned IDs from executed evidence. Completed generation pairs require successful outputs from both models; determinate pairs require both evaluations to be correct/incorrect; unknown pairs have both outputs but at least one nondeterminate evaluation; missing pairs lack a successful output from either model. Determinate, unknown and missing counts partition the planned set.

Latency is CUDA-synchronized generation-only latency; it excludes model download/load, tokenization, queueing, evaluation and export. Throughput uses only samples with both a valid token count and positive latency. Ten examples support descriptive paired counts, not strong population claims.

In [25]:
from experiments.kaggle_local import model_metrics, paired_metrics, paired_rows, write_paired_csv

paired = paired_rows(SAMPLES, small_evaluated, strong_evaluated)
PAIR_METRICS = paired_metrics(SAMPLES, small_evaluated, strong_evaluated)
SMALL_METRICS = model_metrics(small_evaluated)
STRONG_METRICS = model_metrics(strong_evaluated)
PAIRED_COUNTS = PAIR_METRICS["paired_outcome_counts"]
assert sum(PAIRED_COUNTS.values()) == PAIR_METRICS["planned_pair_count"]
write_paired_csv(OUTPUT_DIR / "paired_results.csv", paired)
METRICS = {
    **PAIR_METRICS,
    "small": SMALL_METRICS,
    "strong": STRONG_METRICS,
    "latency_definition": (
        "CUDA-synchronized model.generate elapsed time only; excludes load, tokenization, "
        "queueing, evaluation and export."
    ),
    "throughput_definition": (
        "Output tokens divided by summed positive generation latency for records having "
        "both a valid non-negative integer token count and a finite positive latency."
    ),
    "pair_definitions": {
        "planned_pair_count": "Number of sample IDs in the frozen collection plan.",
        "completed_generation_pair_count": "Pairs where both models have status=completed and a non-empty output string.",
        "determinate_pair_count": "Completed generation pairs with determinate correct/incorrect evaluator status for both models.",
        "unknown_pair_count": "Completed generation pairs where at least one evaluation is missing, unsupported, ambiguous or otherwise not determinate.",
        "missing_pair_count": "Planned pairs where either model lacks a successful generation, including missing, failed, overlength or interrupted outcomes.",
    },
    "uncertainty_note": "Descriptive pilot; no inferential/statistical significance claim from n=10.",
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(METRICS, indent=2, sort_keys=True) + "\n")
print(json.dumps(METRICS, indent=2))

{
  "planned_pair_count": 10,
  "completed_generation_pair_count": 0,
  "determinate_pair_count": 0,
  "unknown_pair_count": 0,
  "missing_pair_count": 10,
  "paired_outcome_counts": {
    "missing_generation": 10
  },
  "small": {
    "attempted_samples": 0,
    "successful_generations": 0,
    "determinately_evaluated_samples": 0,
    "correct_samples": 0,
    "accuracy": null,
    "accuracy_denominator": 0,
    "mean_inference_latency_s": null,
    "median_inference_latency_s": null,
    "p95_inference_latency_s": null,
    "mean_output_tokens": null,
    "total_output_tokens": 0,
    "peak_gpu_allocated_bytes": null,
    "peak_gpu_reserved_bytes": null,
    "tokens_per_second": null,
    "throughput_sample_count": 0,
    "missing_latency_count": 0,
    "missing_token_count": 0,
    "unknown_evaluation_samples": 0,
    "failed_samples": 0,
    "not_executed_samples": 0
  },
  "strong": {
    "attempted_samples": 0,
    "successful_generations": 0,
    "determinately_evaluated_sample

## CELL 13 — Visualization
Generate accuracy, generation-latency, output-token and paired-outcome plots under `outputs/figures`. Empty/disabled runs show an explicit no-inference message rather than invented values.

In [26]:
import matplotlib.pyplot as plt

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
labels = ["Qwen3-1.7B", "Qwen3-8B"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, title, key, suffix in [
    (axes[0], "Accuracy (determinate denominator)", "accuracy", ""),
    (axes[1], "Mean generation latency (s)", "mean_inference_latency_s", ""),
    (axes[2], "Total output tokens", "total_output_tokens", ""),
]:
    values = [SMALL_METRICS[key], STRONG_METRICS[key]]
    if all(value is None for value in values):
        ax.text(0.5, 0.5, "No inference executed", ha="center", va="center")
        ax.set_xticks(range(2), labels)
    else:
        ax.bar(labels, [value or 0 for value in values])
    ax.set_title(title)
fig.tight_layout()
fig.savefig(FIG_DIR / "model_comparison.png", dpi=150)
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 4))
outcome_names = list(PAIRED_COUNTS)
ax.bar(outcome_names, [PAIRED_COUNTS[name] for name in outcome_names])
ax.set_ylabel("Paired samples")
ax.set_title("Paired correctness outcomes")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIG_DIR / "paired_outcomes.png", dpi=150)
plt.close(fig)
print([str(path) for path in sorted(FIG_DIR.glob("*.png"))])

['/kaggle/working/outputs/figures/model_comparison.png', '/kaggle/working/outputs/figures/paired_outcomes.png']


## CELL 14 — Error analysis
Summarize only recorded outputs and evaluator statuses. This is a candidate list for human review, not fabricated qualitative conclusions.

In [27]:
def error_lines(name, records):
    lines = [f"## {name}"]
    for row in records:
        evaluation = row.get("evaluation") or {}
        if row.get("status") != "completed" or evaluation.get("is_correct") is not True:
            lines += [
                f"### {row['sample_id']} — execution={row.get('status')}, evaluator={evaluation.get('status', 'not evaluated')}",
                f"- Error: {row.get('error_type')}: {row.get('error_message')}",
                f"- Output: {row.get('model_output')}",
                f"- Evaluator reason: {evaluation.get('reason')}",
            ]
    return "\n".join(lines)


ERROR_ANALYSIS = (
    "# Error analysis\n\n"
    + error_lines("Qwen3-1.7B", small_evaluated)
    + "\n\n"
    + error_lines("Qwen3-8B", strong_evaluated)
)
(OUTPUT_DIR / "error_analysis.md").write_text(ERROR_ANALYSIS + "\n")
print(ERROR_ANALYSIS[:5000])

# Error analysis

## Qwen3-1.7B

## Qwen3-8B


## CELL 15 — Export outputs
Write model revisions and provenance, then package only generated results and figures. The ZIP excludes HF/model caches, credentials, and weights. The append-only execution checkpoints stay outside the archive for resumability.

In [28]:
import zipfile

from experiments.kaggle_local import (
    notebook_scientific_status,
    read_records,
    sample_ids_sha256,
)

try:
    import psutil

    ram_meta = {
        "total_bytes": psutil.virtual_memory().total,
        "available_bytes": psutil.virtual_memory().available,
    }
except ImportError:
    ram_meta = None
ENVIRONMENT = {
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "gpu": GPU_BASELINE,
    "cpu_ram": ram_meta,
    "repository_commit": REPO_COMMIT,
}
REVISIONS = {
    "small": {"model_id": SMALL_MODEL_ID, "revision": SMALL_MODEL_REVISION, "dtype": "float16"},
    "strong": {
        "model_id": STRONG_MODEL_ID,
        "revision": STRONG_MODEL_REVISION,
        "quantization": {"load_in_4bit": True, "type": "nf4", "compute_dtype": "float16"},
    },
}
# Persist environment/model evidence before allowing it to contribute to status.
(OUTPUT_DIR / "environment.json").write_text(
    json.dumps(ENVIRONMENT, indent=2, sort_keys=True) + "\n"
)
(OUTPUT_DIR / "model_revisions.json").write_text(
    json.dumps(REVISIONS, indent=2, sort_keys=True) + "\n"
)
expected_status_provenance = {
    "dataset_revision": DATASET_MANIFEST.revision,
    "dataset_content_sha256": DATASET_MANIFEST.dataset_content_sha256,
    "collection_plan_sha256": COLLECTION_PLAN["plan_sha256"],
    "prompt_version": PROMPT_VERSION,
    "original_prompt_sha256": PROMPT_SHA256,
}
try:
    full_run_marker = json.loads((CHECKPOINT_DIR / "full_run_started.json").read_text("utf-8"))
except FileNotFoundError:
    full_run_marker = None
status_small_full = read_records(CHECKPOINT_DIR / "small_full.jsonl")
status_strong_full = read_records(CHECKPOINT_DIR / "strong_full.jsonl")
status_small_smoke = read_records(CHECKPOINT_DIR / "small_smoke.jsonl")
status_strong_smoke = read_records(CHECKPOINT_DIR / "strong_smoke.jsonl")
revision_evidence = json.loads((OUTPUT_DIR / "model_revisions.json").read_text("utf-8"))
environment_evidence = json.loads((OUTPUT_DIR / "environment.json").read_text("utf-8"))
required_evidence = (
    DATASET_MANIFEST.dataset_content_sha256
    == "aa75c686332c09e12ce6412571b32b7efae779dfc8696d3448b29a45e5542e4b"
    and COLLECTION_PLAN["plan_sha256"]
    == "1d5065acc16b50c1521422ec2a912ed28a10bd483d905054fbdd24aefd642493"
    and sample_ids_sha256(SAMPLE_IDS)
    == "077275a61e532506eab2d10c09d36960976253969380ca256af7b95cbcced5ed"
    and len(SAMPLE_IDS) == 10
    and all(
        (OUTPUT_DIR / required_file).is_file()
        for required_file in (
            "small_results.jsonl",
            "strong_results.jsonl",
            "paired_results.csv",
            "metrics.json",
        )
    )
    and torch.cuda.is_available()
    and "t4" in environment_evidence.get("gpu", {}).get("name", "").lower()
    and revision_evidence["small"]["model_id"] == SMALL_MODEL_ID
    and revision_evidence["small"]["revision"] == SMALL_MODEL_REVISION
    and revision_evidence["strong"]["model_id"] == STRONG_MODEL_ID
    and revision_evidence["strong"]["revision"] == STRONG_MODEL_REVISION
    and all(
        environment_evidence.get(name)
        for name in ("torch", "cuda", "transformers", "accelerate", "bitsandbytes")
    )
)
SCIENTIFIC_STATUS = notebook_scientific_status(
    SAMPLES,
    status_small_full,
    status_strong_full,
    status_small_smoke,
    status_strong_smoke,
    small_model_id=SMALL_MODEL_ID,
    small_model_revision=SMALL_MODEL_REVISION,
    strong_model_id=STRONG_MODEL_ID,
    strong_model_revision=STRONG_MODEL_REVISION,
    expected_provenance=expected_status_provenance,
    required_evidence=required_evidence,
    full_run_marker=full_run_marker,
)
METRICS["scientific_status"] = SCIENTIFIC_STATUS
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(METRICS, indent=2, sort_keys=True) + "\n")
RUN_MANIFEST = {
    "schema_version": "kaggle-qwen3-pilot-manifest-v1",
    "provider": "local_transformers",
    "scientific_eligible": False,
    "execution_mode": "offline_or_local_gpu",
    "run_full_pilot": RUN_FULL_PILOT,
    "scientific_status": SCIENTIFIC_STATUS,
    "dataset": DATASET_MANIFEST.dataset,
    "dataset_revision": DATASET_MANIFEST.revision,
    "dataset_content_sha256": DATASET_MANIFEST.dataset_content_sha256,
    "collection_plan_sha256": COLLECTION_PLAN["plan_sha256"],
    "split": COLLECTION_PLAN["split"],
    "sample_ids": SAMPLE_IDS,
    "sample_ids_sha256": sample_ids_sha256(SAMPLE_IDS),
    "sample_count": len(SAMPLE_IDS),
    "prompt_version": PROMPT_VERSION,
    "prompt_sha256": PROMPT_SHA256,
    "evaluator_version": EVALUATOR_VERSION,
    "generation": {
        "batch_size": 1,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "seed": SEED,
        "do_sample": False,
        "enable_thinking": False,
        "retry": False,
        "fallback": False,
    },
    "latency_scope": METRICS["latency_definition"],
    "repository_commit": REPO_COMMIT,
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(RUN_MANIFEST, indent=2, sort_keys=True) + "\n"
)
zip_path = Path("/kaggle/working/outputs.zip")
include = [
    "small_results.jsonl",
    "strong_results.jsonl",
    "paired_results.csv",
    "metrics.json",
    "environment.json",
    "model_revisions.json",
    "run_manifest.json",
    "error_analysis.md",
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in include:
        archive.write(OUTPUT_DIR / name, arcname=name)
    for figure in FIG_DIR.glob("*.png"):
        archive.write(figure, arcname=f"figures/{figure.name}")
print({"output_dir": str(OUTPUT_DIR), "zip": str(zip_path), "zip_files": include})

{'output_dir': '/kaggle/working/outputs', 'zip': '/kaggle/working/outputs.zip', 'zip_files': ['small_results.jsonl', 'strong_results.jsonl', 'paired_results.csv', 'metrics.json', 'environment.json', 'model_revisions.json', 'run_manifest.json', 'error_analysis.md']}


## CELL 16 — Final summary
Scientific status is `NOTEBOOK_READY_FOR_KAGGLE` until there is actual Kaggle GPU evidence. Smoke/pilot status is never inferred from configuration or notebook execution without successful generation records.

In [29]:
def peak(records, key):
    return max((row.get(key, 0) or 0 for row in records), default=None)


summary = {
    "scientific_status": SCIENTIFIC_STATUS,
    "gpu": GPU_BASELINE["name"],
    "models": REVISIONS,
    "dataset_validated": True,
    "sample_count": len(SAMPLES),
    "planned_pair_count": PAIR_METRICS["planned_pair_count"],
    "completed_generation_pair_count": PAIR_METRICS["completed_generation_pair_count"],
    "determinate_pair_count": PAIR_METRICS["determinate_pair_count"],
    "unknown_pair_count": PAIR_METRICS["unknown_pair_count"],
    "missing_pair_count": PAIR_METRICS["missing_pair_count"],
    "paired_outcome_counts": PAIRED_COUNTS,
    "accuracy": {
        "small": f"{SMALL_METRICS['correct_samples']}/{SMALL_METRICS['accuracy_denominator']}",
        "strong": f"{STRONG_METRICS['correct_samples']}/{STRONG_METRICS['accuracy_denominator']}",
    },
    "mean_generation_latency_s": {
        "small": SMALL_METRICS["mean_inference_latency_s"],
        "strong": STRONG_METRICS["mean_inference_latency_s"],
    },
    "total_output_tokens": {
        "small": SMALL_METRICS["total_output_tokens"],
        "strong": STRONG_METRICS["total_output_tokens"],
    },
    "peak_vram_allocated_bytes": {
        "small": peak(small_evaluated, "peak_gpu_allocated_bytes"),
        "strong": peak(strong_evaluated, "peak_gpu_allocated_bytes"),
    },
    "failed_samples": {
        "small": SMALL_METRICS["failed_samples"],
        "strong": STRONG_METRICS["failed_samples"],
    },
    "outputs": str(OUTPUT_DIR),
    "archive": str(zip_path),
}
print(json.dumps(summary, indent=2))

{
  "scientific_status": "NOTEBOOK_READY_FOR_KAGGLE",
  "gpu": "Tesla T4",
  "models": {
    "small": {
      "model_id": "Qwen/Qwen3-1.7B",
      "revision": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
      "dtype": "float16"
    },
    "strong": {
      "model_id": "Qwen/Qwen3-8B",
      "revision": "b968826d9c46dd6066d109eabc6255188de91218",
      "quantization": {
        "load_in_4bit": true,
        "type": "nf4",
        "compute_dtype": "float16"
      }
    }
  },
  "dataset_validated": true,
  "sample_count": 10,
  "planned_pair_count": 10,
  "completed_generation_pair_count": 0,
  "determinate_pair_count": 0,
  "unknown_pair_count": 0,
  "missing_pair_count": 10,
  "paired_outcome_counts": {
    "missing_generation": 10
  },
  "accuracy": {
    "small": "0/0",
    "strong": "0/0"
  },
  "mean_generation_latency_s": {
    "small": null,
    "strong": null
  },
  "total_output_tokens": {
    "small": 0,
    "strong": 0
  },
  "peak_vram_allocated_bytes": {
    "small": null,
